# AstroCLIP Teaching Lab (Standalone)

This notebook walks through a compact AstroCLIP pipeline:

1. Load a manageable subset of the hosted `EiffL/AstroCLIP` dataset.
2. Train lightweight image and spectrum autoencoders entirely within the notebook.
3. Align both modalities with a CLIP loss.
4. Embed validation samples and explore the joint space.
5. Perform a simple downstream redshift regression.

Everything runs locally; no external scripts are required. Adjust hyper-parameters for longer or shorter runs.

## 0. Environment Prerequisites

Ensure you are running inside the `astroclip-teaching` (or mac equivalent) environment. Required packages: `datasets`, `torch`, `lightning`, `matplotlib`, `umap-learn`, `pandas`, `h5py`.

Set environment variables beforehand, e.g.:

```bash
export ASTROCLIP_ROOT=/path/to/astroclip_data   # adjust as needed
export HF_DATASETS_CACHE=/path/for/hf_cache     # optional, recommended for large downloads
```

The notebook below assumes the hosted dataset `EiffL/AstroCLIP` is accessible (or already cached).

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

import torch
import lightning as L
from lightning.pytorch.callbacks import ModelCheckpoint

from datasets import load_dataset, load_from_disk, DatasetDict

import h5py
import umap

from astroclip.data.datamodule import AstroClipCollator

from teaching_scripts.data_utils import (
    load_astroclip_dataset,
    build_image_dataloader,
    build_spectrum_dataloader,
    build_multimodal_dataloader,
)
from teaching_scripts.models import (
    ImageAutoencoder,
    SpectrumAutoencoder,
    SmallCLIPModel,
)

ASTROCLIP_ROOT = Path(os.environ.get('ASTROCLIP_ROOT', './astroclip_demo')).resolve()
ASTROCLIP_ROOT.mkdir(parents=True, exist_ok=True)
DATASET_PATH = ASTROCLIP_ROOT / 'teaching_notebook_subset'
print(f"ASTROCLIP_ROOT: {ASTROCLIP_ROOT}")
print(f"Dataset path:  {DATASET_PATH}")

### Load Subset

Provide the path to a dataset prepared with `teaching_scripts/prepare_dataset.py` or `load_dataset(...).save_to_disk(...)`. Set `DATASET_PATH` accordingly.

In [ ]:
# Adjust this path to the existing subset
DATASET_PATH = ASTROCLIP_ROOT / 'teaching_notebook_subset'

if not DATASET_PATH.exists():
    raise FileNotFoundError(f"{DATASET_PATH} not found. Prepare the subset first.")

ds = load_from_disk(DATASET_PATH)
print(ds)
display({split: len(ds[split]) for split in ds})

### Inspect Sample

Visualise a few image/spectrum pairs to verify data integrity.

In [ ]:
collator = AstroClipCollator(center_crop=144)

samples = []
for i in range(4):
    item = dict(ds['train'][i])
    item['image'] = np.asarray(item['image'])
    item['spectrum'] = np.asarray(item['spectrum'])
    samples.append(item)

batch = collator(samples)
images = batch['image']
spectra_raw = [np.asarray(s) for s in batch['spectrum']]
print('Image batch shape:', images.shape)
print('Raw spectrum shape example:', spectra_raw[0].shape)

spectra = np.stack([spec.reshape(-1) for spec in spectra_raw])
print('Flattened spectra shape:', spectra.shape)

fig, axes = plt.subplots(1, 4, figsize=(12, 4))
for ax, img in zip(axes, images):
    ax.imshow(img.permute(1, 2, 0).numpy())
    ax.axis('off')
plt.show()

plt.figure(figsize=(8, 3))
plt.plot(spectra[0])
plt.title('Example spectrum')
plt.xlabel('Pixel')
plt.ylabel('Flux')
plt.show()


## 1. Train Image Autoencoder

Use a lightweight convolutional autoencoder to encode 144×144 RGB crops. Training is intentionally short; increase `max_epochs` or model size for better reconstructions.

In [ ]:
from lightning.pytorch.loggers import CSVLogger

image_logger = CSVLogger(save_dir=str(ASTROCLIP_ROOT / 'logs'), name='image_autoencoder_notebook')
image_train_loader = build_image_dataloader(ds['train'], batch_size=128, shuffle=True, num_workers=0)
image_val_loader = build_image_dataloader(ds['test'], batch_size=128, shuffle=False, num_workers=0)

image_model = ImageAutoencoder(embed_dim=256)
checkpoint_img = ModelCheckpoint(dirpath=ASTROCLIP_ROOT, filename='image_autoencoder_notebook', save_last=True, save_top_k=1)
trainer_img = L.Trainer(accelerator='auto', devices='auto', max_epochs=8, log_every_n_steps=25, callbacks=[checkpoint_img], logger=image_logger)
trainer_img.fit(image_model, image_train_loader, image_val_loader)
image_ckpt_path = ASTROCLIP_ROOT / 'image_autoencoder_notebook.ckpt'
trainer_img.save_checkpoint(image_ckpt_path)
print(f"Saved image autoencoder checkpoint: {image_ckpt_path}")

In [ ]:
# Visualise image reconstruction
image_model.eval()
with torch.no_grad():
    batch_imgs = next(iter(image_val_loader))
    recon_imgs = image_model(batch_imgs.to(image_model.device)).cpu()

fig, axes = plt.subplots(2, min(4, len(batch_imgs)), figsize=(12, 6))
for i in range(min(4, len(batch_imgs))):
    axes[0, i].imshow(batch_imgs[i].permute(1, 2, 0).numpy())
    axes[0, i].axis('off')
    axes[1, i].imshow(recon_imgs[i].permute(1, 2, 0).numpy())
    axes[1, i].axis('off')
axes[0,0].set_title('Originals')
axes[1,0].set_title('Reconstructions')
plt.show()


In [ ]:
history_img = trainer_img.logged_metrics
print("Final logged metrics:", history_img)

train_hist_img = trainer_img.callback_metrics.get('train_loss')
val_hist_img = trainer_img.callback_metrics.get('val_loss')

In [ ]:
metrics_img = trainer_img.logger.experiment.metrics if hasattr(trainer_img.logger, 'experiment') else []
if metrics_img:
    # When using the default logger the metrics list may be empty; fallback to manual history if available
    print(metrics_img[:3])

Plot the loss curve from the trainer logs.

In [ ]:
import pandas as pd
from pathlib import Path

metrics_path_img = Path(image_logger.log_dir) / 'metrics.csv'
if metrics_path_img.exists():
    df_img = pd.read_csv(metrics_path_img)
else:
    df_img = pd.DataFrame()

if not df_img.empty:
    plt.figure(figsize=(6, 4))
    if 'train_loss' in df_img.columns:
        plt.plot(df_img['epoch'], df_img['train_loss'], label='train_loss')
    if 'val_loss' in df_img.columns:
        plt.plot(df_img['epoch'], df_img['val_loss'], label='val_loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Image AE loss curve')
    plt.legend()
    plt.show()
else:
    print('No metrics recorded for plotting. Check logger output.')

## 2. Train Spectrum Autoencoder

A simple fully-connected autoencoder for the 1D spectra.

In [ ]:
from lightning.pytorch.loggers import CSVLogger as CSVLoggerSpec

spectrum_logger = CSVLoggerSpec(save_dir=str(ASTROCLIP_ROOT / 'logs'), name='spectrum_autoencoder_notebook')
spectrum_train_loader = build_spectrum_dataloader(ds['train'], batch_size=256, shuffle=True, num_workers=0)
spectrum_val_loader = build_spectrum_dataloader(ds['test'], batch_size=256, shuffle=False, num_workers=0)

sample_spec = ds['train'][0]['spectrum']
input_dim = int(np.prod(np.asarray(sample_spec).shape))

spectrum_model = SpectrumAutoencoder(input_dim=input_dim, embed_dim=256)
checkpoint_spec = ModelCheckpoint(dirpath=ASTROCLIP_ROOT, filename='spectrum_autoencoder_notebook', save_last=True, save_top_k=1)
trainer_spec = L.Trainer(accelerator='auto', devices='auto', max_epochs=12, log_every_n_steps=25, callbacks=[checkpoint_spec], logger=spectrum_logger)
trainer_spec.fit(spectrum_model, spectrum_train_loader, spectrum_val_loader)
spectrum_ckpt_path = ASTROCLIP_ROOT / 'spectrum_autoencoder_notebook.ckpt'
trainer_spec.save_checkpoint(spectrum_ckpt_path)
print(f"Saved spectrum autoencoder checkpoint: {spectrum_ckpt_path}")

In [ ]:
# Visualise spectrum reconstruction
spectrum_model.eval()
with torch.no_grad():
    batch_spec = next(iter(spectrum_val_loader))
    recon_spec = spectrum_model(batch_spec.to(spectrum_model.device)).cpu()

plt.figure(figsize=(10, 4))
plt.plot(batch_spec[0].numpy(), label='Original')
plt.plot(recon_spec[0].numpy(), label='Reconstruction')
plt.title('Spectrum reconstruction example')
plt.xlabel('Pixel')
plt.ylabel('Flux')
plt.legend()
plt.show()


In [ ]:
metrics_path_spec = Path(spectrum_logger.log_dir) / 'metrics.csv'
if metrics_path_spec.exists():
    df_spec = pd.read_csv(metrics_path_spec)
else:
    df_spec = pd.DataFrame()

if not df_spec.empty:
    plt.figure(figsize=(6, 4))
    if 'train_loss' in df_spec.columns:
        plt.plot(df_spec['epoch'], df_spec['train_loss'], label='train_loss')
    if 'val_loss' in df_spec.columns:
        plt.plot(df_spec['epoch'], df_spec['val_loss'], label='val_loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Spectrum AE loss curve')
    plt.legend()
    plt.show()
else:
    print('No metrics recorded for plotting. Check logger output.')

## 3. CLIP Alignment

Load the two autoencoders and train the CLIP objective for a few epochs.

In [ ]:
from lightning.pytorch.loggers import CSVLogger as CSVLoggerClip

clip_logger = CSVLoggerClip(save_dir=str(ASTROCLIP_ROOT / 'logs'), name='clip_alignment_notebook')
clip_train_loader = build_multimodal_dataloader(ds['train'], batch_size=256, shuffle=True, num_workers=0)
clip_val_loader = build_multimodal_dataloader(ds['test'], batch_size=256, shuffle=False, num_workers=0)

image_encoder = ImageAutoencoder.load_from_checkpoint(image_ckpt_path)
spectrum_encoder = SpectrumAutoencoder.load_from_checkpoint(spectrum_ckpt_path)

clip_model = SmallCLIPModel(
    image_encoder=image_encoder,
    spectrum_encoder=spectrum_encoder,
    projection_dim=256,
    lr=5e-4,
    temperature=0.1,
    finetune_encoders=False,
)

checkpoint_clip = ModelCheckpoint(dirpath=ASTROCLIP_ROOT, filename='clip_alignment_notebook', save_last=True, save_top_k=1)
trainer_clip = L.Trainer(accelerator='auto', devices='auto', max_epochs=10, log_every_n_steps=25, callbacks=[checkpoint_clip], logger=clip_logger)
trainer_clip.fit(clip_model, clip_train_loader, clip_val_loader)
clip_ckpt_path = ASTROCLIP_ROOT / 'clip_alignment_notebook.ckpt'
trainer_clip.save_checkpoint(clip_ckpt_path)
print(f"Saved CLIP alignment checkpoint: {clip_ckpt_path}")

In [ ]:
metrics_path_clip = Path(clip_logger.log_dir) / 'metrics.csv'
if metrics_path_clip.exists():
    df_clip = pd.read_csv(metrics_path_clip)
else:
    df_clip = pd.DataFrame()

if not df_clip.empty:
    fig, ax = plt.subplots(figsize=(6, 4))
    if 'train_loss' in df_clip.columns:
        ax.plot(df_clip['epoch'], df_clip['train_loss'], label='train_loss')
    if 'val_loss' in df_clip.columns:
        ax.plot(df_clip['epoch'], df_clip['val_loss'], label='val_loss')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.set_title('CLIP alignment loss curve')
    ax.legend(loc='upper left')
    if 'logit_scale' in df_clip.columns:
        ax2 = ax.twinx()
        ax2.plot(df_clip['epoch'], df_clip['logit_scale'], color='tab:green', linestyle='--', label='logit_scale')
        ax2.set_ylabel('Logit scale')
        ax2.legend(loc='lower right')
    plt.show()
else:
    print('No metrics recorded for plotting. Check logger output.')

## 4. Embed Validation Split

Generate embeddings for the test split and store them for later use.

In [ ]:
clip_model.eval()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
clip_model.to(device)

val_loader = build_multimodal_dataloader(ds['test'], batch_size=256, shuffle=False, num_workers=4)

image_embeddings = []
spectrum_embeddings = []
redshifts = []
targetids = []

with torch.no_grad():
    for batch in val_loader:
        images = batch['image'].to(device)
        spectra = batch['spectrum'].to(device)
        img_emb, sp_emb = clip_model(images, spectra)
        image_embeddings.append(img_emb.cpu().numpy())
        spectrum_embeddings.append(sp_emb.cpu().numpy())
        if 'redshift' in batch:
            redshifts.append(batch['redshift'].numpy())
        if 'targetid' in batch:
            targetids.append(batch['targetid'].numpy())

image_embeddings = np.concatenate(image_embeddings, axis=0)
spectrum_embeddings = np.concatenate(spectrum_embeddings, axis=0)
redshift = np.concatenate(redshifts, axis=0) if redshifts else None
targetid = np.concatenate(targetids, axis=0) if targetids else None

print(image_embeddings.shape, spectrum_embeddings.shape)


### UMAP Projection

Visualise the joint embedding space.

In [ ]:
reducer = umap.UMAP(random_state=42)
joint = reducer.fit_transform(np.concatenate([image_embeddings, spectrum_embeddings], axis=0))
labels = np.concatenate([np.zeros(len(image_embeddings)), np.ones(len(spectrum_embeddings))])

plt.figure(figsize=(6, 5))
plt.scatter(joint[labels == 0, 0], joint[labels == 0, 1], s=5, alpha=0.5, label='Images')
plt.scatter(joint[labels == 1, 0], joint[labels == 1, 1], s=5, alpha=0.5, label='Spectra')
plt.legend()
plt.title('UMAP of joint embedding space')
plt.show()

## 5. Downstream Redshift Regression

Train a tiny MLP on the embeddings.

In [ ]:
from torch.utils.data import DataLoader, TensorDataset

features = np.concatenate([image_embeddings, spectrum_embeddings], axis=1)
labels = redshift if redshift is not None else np.zeros(len(features))

split = int(0.8 * len(features))
train_X, val_X = features[:split], features[split:]
train_y, val_y = labels[:split], labels[split:]

train_ds = TensorDataset(torch.from_numpy(train_X).float(), torch.from_numpy(train_y).float())
val_ds = TensorDataset(torch.from_numpy(val_X).float(), torch.from_numpy(val_y).float())
train_dl = DataLoader(train_ds, batch_size=128, shuffle=True)
val_dl = DataLoader(val_ds, batch_size=128)

dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class MLP(torch.nn.Module):
    def __init__(self, input_dim, hidden=256):
        super().__init__()
        self.net = torch.nn.Sequential(
            torch.nn.Linear(input_dim, hidden),
            torch.nn.ReLU(),
            torch.nn.Linear(hidden, hidden // 2),
            torch.nn.ReLU(),
            torch.nn.Linear(hidden // 2, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)

model = MLP(features.shape[1]).to(dev)
criterion = torch.nn.MSELoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-5)

train_losses, val_losses = [], []
for epoch in range(50):
    model.train()
    running = 0.0
    for batch_x, batch_y in train_dl:
        batch_x, batch_y = batch_x.to(dev), batch_y.to(dev)
        optimizer.zero_grad(set_to_none=True)
        preds = model(batch_x)
        loss = criterion(preds, batch_y)
        loss.backward()
        optimizer.step()
        running += loss.item() * batch_x.size(0)
    train_loss = running / len(train_dl.dataset)

    model.eval()
    running = 0.0
    with torch.no_grad():
        for batch_x, batch_y in val_dl:
            batch_x, batch_y = batch_x.to(dev), batch_y.to(dev)
            preds = model(batch_x)
            loss = criterion(preds, batch_y)
            running += loss.item() * batch_x.size(0)
    val_loss = running / len(val_dl.dataset)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:02d}: train_loss={train_loss:.4f}, val_loss={val_loss:.4f}")

plt.figure(figsize=(6, 4))
plt.plot(train_losses, label='train')
plt.plot(val_losses, label='val')
plt.xlabel('Epoch')
plt.ylabel('MSE')
plt.title('Redshift regression losses')
plt.legend()
plt.show()


In [ ]:
# Predicted vs. true redshift
model.eval()
preds = []
with torch.no_grad():
    for batch_x, _ in val_dl:
        preds.extend(model(batch_x.to(dev)).cpu().numpy())
preds = np.array(preds)
true = val_y[: len(preds)]

plt.figure(figsize=(5, 5))
plt.scatter(true, preds, s=8, alpha=0.6)
lims = [true.min(), true.max()]
plt.plot(lims, lims, '--', color='gray')
plt.xlabel('True redshift')
plt.ylabel('Predicted redshift')
plt.title('Redshift predictions')
plt.show()


## 6. Recap

We trained image and spectrum encoders from scratch, aligned them with a CLIP objective, visualised the joint space, and ran a simple regression head—all inside this notebook. For deeper dives:

- Increase subset size / epochs for better performance.
- Replace autoencoders with the full AstroDINO/SpecFormer models.
- Pull in Galaxy Zoo morphology labels once RA/Dec bridges are available.

Happy experimenting!

## 6. Morphology Classification (Embeddings)

To keep things lightweight we assume you have an HDF5 file containing AstroCLIP image embeddings alongside Galaxy Zoo label fractions (as prepared in the official tutorial). You can generate this file with the toy model in this notebook or copy the one from the HPC pipeline.

We treat the debiased probability of a strong bar (`bar_strong_debiased`) as a binary target (threshold at 0.5). Adjust the column to explore other morphology questions.

In [ ]:
import h5py
import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset

MORPH_EMB_PATH = ASTROCLIP_ROOT / 'galaxy_zoo_embeddings.h5'  # update with actual file
label_column = 'bar_strong_debiased'

if not MORPH_EMB_PATH.exists():
    raise FileNotFoundError(f"Update MORPH_EMB_PATH. File not found: {MORPH_EMB_PATH}")

with h5py.File(MORPH_EMB_PATH, 'r') as f:
    morph_embeddings = f['image_embeddings'][:]
    morph_labels = f[label_column][:]

print('Embeddings shape:', morph_embeddings.shape)
print('Label vector shape:', morph_labels.shape)


In [ ]:
# Build train/val splits
morph_labels_binary = (morph_labels >= 0.5).astype(np.float32)
split_idx = int(0.8 * len(morph_embeddings))

train_X_morph = morph_embeddings[:split_idx]
val_X_morph = morph_embeddings[split_idx:]
train_y_morph = morph_labels_binary[:split_idx]
val_y_morph = morph_labels_binary[split_idx:]

train_ds_morph = TensorDataset(torch.from_numpy(train_X_morph).float(), torch.from_numpy(train_y_morph).float())
val_ds_morph = TensorDataset(torch.from_numpy(val_X_morph).float(), torch.from_numpy(val_y_morph).float())
train_dl_morph = DataLoader(train_ds_morph, batch_size=128, shuffle=True)
val_dl_morph = DataLoader(val_ds_morph, batch_size=128, shuffle=False)


In [ ]:
# Simple binary classifier
morph_model = MLP(train_X_morph.shape[1]).to(dev)  # reuse MLP defined earlier
criterion_bce = torch.nn.BCEWithLogitsLoss()
optimizer_morph = torch.optim.AdamW(morph_model.parameters(), lr=3e-4, weight_decay=1e-5)

train_losses_morph, val_losses_morph = [], []
for epoch in range(30):
    morph_model.train()
    running = 0.0
    for x_batch, y_batch in train_dl_morph:
        x_batch, y_batch = x_batch.to(dev), y_batch.to(dev)
        optimizer_morph.zero_grad()
        logits = morph_model(x_batch)
        loss = criterion_bce(logits, y_batch)
        loss.backward()
        optimizer_morph.step()
        running += loss.item() * x_batch.size(0)
    train_loss = running / len(train_dl_morph.dataset)

    morph_model.eval()
    running = 0.0
    with torch.no_grad():
        for x_batch, y_batch in val_dl_morph:
            x_batch, y_batch = x_batch.to(dev), y_batch.to(dev)
            logits = morph_model(x_batch)
            loss = criterion_bce(logits, y_batch)
            running += loss.item() * x_batch.size(0)
    val_loss = running / len(val_dl_morph.dataset)
    train_losses_morph.append(train_loss)
    val_losses_morph.append(val_loss)

plt.figure(figsize=(6, 4))
plt.plot(train_losses_morph, label='train')
plt.plot(val_losses_morph, label='val')
plt.xlabel('Epoch')
plt.ylabel('BCE loss')
plt.title('Morphology classification losses')
plt.legend()
plt.show()


In [ ]:
from sklearn.metrics import accuracy_score, f1_score

morph_model.eval()
preds, targets = [], []
with torch.no_grad():
    for x_batch, y_batch in val_dl_morph:
        logits = morph_model(x_batch.to(dev))
        preds.extend(torch.sigmoid(logits).cpu().numpy())
        targets.extend(y_batch.numpy())

preds = np.array(preds)
binary_preds = (preds >= 0.5).astype(np.float32)
accuracy = accuracy_score(targets, binary_preds)
f1 = f1_score(targets, binary_preds)
print(f"Accuracy: {accuracy:.3f}, F1: {f1:.3f}")

plt.figure(figsize=(5, 5))
plt.scatter(targets, preds, s=8, alpha=0.6)
plt.xlabel('True label (probability)')
plt.ylabel('Predicted probability')
plt.title('Morphology predictions')
plt.show()
